# 第二章：State 设计

## 学习目标
- 理解 State 的默认更新行为（覆盖）
- 用 `Annotated` + reducer 实现追加、去重等自定义更新策略
- 掌握内置 `add` reducer 和自定义 reducer 函数
- 使用 LangGraph 内置的 `MessagesState` 快速构建对话应用
- 建立 State 设计的最佳实践

## 0. 环境准备

In [ ]:
from importlib.metadata import version
print(f"langgraph 版本: {version('langgraph')}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

## 1. 默认行为：覆盖

上一章我们用 `TypedDict` 定义 State，节点返回的 dict 会**覆盖**对应字段。先确认这个行为：

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 默认行为：每个字段都是覆盖
class OverwriteState(TypedDict):
    items: list[str]

def node_a(state: OverwriteState) -> dict:
    return {"items": ["a1", "a2"]}

def node_b(state: OverwriteState) -> dict:
    return {"items": ["b1"]}

graph = StateGraph(OverwriteState)
graph.add_node("a", node_a)
graph.add_node("b", node_b)
graph.add_edge(START, "a")
graph.add_edge("a", "b")
graph.add_edge("b", END)

app = graph.compile()
result = app.invoke({"items": []})
print(result["items"])  # 只有 b1，a 的结果被覆盖了

### 问题在哪？

节点 `a` 写入了 `["a1", "a2"]`，节点 `b` 写入了 `["b1"]`。最终结果只有 `["b1"]`——`a` 的数据丢了。

对于像对话消息这样需要**累积**的数据，覆盖行为显然不对。我们需要的是「追加」而不是「覆盖」。

## 2. Annotated + reducer：自定义更新策略

LangGraph 用 `Annotated[类型, reducer函数]` 语法来指定字段的更新方式。`reducer` 是一个函数，接收旧值和新值，返回合并后的结果。

In [ ]:
from typing import Annotated
from operator import add  # add 对列表就是拼接：[1,2] + [3] = [1,2,3]

# 使用 Annotated + add reducer
class AppendState(TypedDict):
    items: Annotated[list[str], add]  # 追加而非覆盖

def node_a(state: AppendState) -> dict:
    return {"items": ["a1", "a2"]}

def node_b(state: AppendState) -> dict:
    return {"items": ["b1"]}

graph = StateGraph(AppendState)
graph.add_node("a", node_a)
graph.add_node("b", node_b)
graph.add_edge(START, "a")
graph.add_edge("a", "b")
graph.add_edge("b", END)

app = graph.compile()
result = app.invoke({"items": []})
print(result["items"])  # ["a1", "a2", "b1"] —— 追加成功

### 刚才发生了什么？

唯一的变化是 State 定义从：
```python
items: list[str]               # 覆盖
```
改成了：
```python
items: Annotated[list[str], add]  # 追加
```

`operator.add` 对列表的行为就是拼接（`[1,2] + [3]` = `[1,2,3]`）。LangGraph 在每个节点返回后，不再直接覆盖，而是调用 `add(旧值, 新值)` 来合并。

| 语法 | 更新行为 | 等价操作 |
|------|---------|----------|
| `items: list` | 覆盖 | `state["items"] = new_value` |
| `items: Annotated[list, add]` | 追加 | `state["items"] = add(old_value, new_value)` |

## 3. 自定义 reducer 函数

`add` 只是最简单的 reducer。你可以写任何函数，只要签名是 `(old, new) -> merged`。

In [ ]:
# 自定义 reducer：追加并去重
def add_unique(old: list[str], new: list[str]) -> list[str]:
    """追加新元素，但跳过已存在的"""
    result = old.copy()
    for item in new:
        if item not in result:
            result.append(item)
    return result

class UniqueState(TypedDict):
    tags: Annotated[list[str], add_unique]

def node_a(state: UniqueState) -> dict:
    return {"tags": ["python", "langchain"]}

def node_b(state: UniqueState) -> dict:
    return {"tags": ["langchain", "langgraph"]}  # "langchain" 重复

graph = StateGraph(UniqueState)
graph.add_node("a", node_a)
graph.add_node("b", node_b)
graph.add_edge(START, "a")
graph.add_edge("a", "b")
graph.add_edge("b", END)

app = graph.compile()
result = app.invoke({"tags": []})
print(result["tags"])  # ["python", "langchain", "langgraph"] —— 无重复

### reducer 函数的规则

| 要求 | 说明 |
|------|------|
| 签名 | `(old_value, new_value) -> merged_value` |
| 返回类型 | 必须和字段类型一致 |
| 无副作用 | 不要修改 `old_value`，创建新对象返回 |

常见 reducer 模式：

```python
# 追加
Annotated[list, add]             # operator.add

# 追加并去重
Annotated[list, add_unique]      # 自定义

# 只保留最新 N 条
def keep_last_n(old, new, n=5):
    return (old + new)[-n:]

# 计数器累加
Annotated[int, add]              # 1 + 1 = 2
```

## 4. 混合使用：部分覆盖，部分追加

实际项目中，不同字段往往需要不同的更新策略。没有 `Annotated` 的字段保持覆盖行为。

In [ ]:
class MixedState(TypedDict):
    messages: Annotated[list[str], add]  # 追加
    status: str                          # 覆盖
    step_count: Annotated[int, add]      # 累加

def step_one(state: MixedState) -> dict:
    return {
        "messages": ["步骤一完成"],
        "status": "processing",
        "step_count": 1,
    }

def step_two(state: MixedState) -> dict:
    return {
        "messages": ["步骤二完成"],
        "status": "done",
        "step_count": 1,
    }

graph = StateGraph(MixedState)
graph.add_node("step_one", step_one)
graph.add_node("step_two", step_two)
graph.add_edge(START, "step_one")
graph.add_edge("step_one", "step_two")
graph.add_edge("step_two", END)

app = graph.compile()
result = app.invoke({"messages": [], "status": "init", "step_count": 0})

print(f"messages: {result['messages']}")      # 追加：两条都在
print(f"status: {result['status']}")           # 覆盖：只有最后的 done
print(f"step_count: {result['step_count']}")   # 累加：0 + 1 + 1 = 2

### 设计思路

选择覆盖还是追加，取决于字段的语义：

| 字段语义 | 更新策略 | 例子 |
|---------|---------|------|
| 历史记录、日志 | 追加 (`add`) | 对话消息、操作记录 |
| 当前状态、标志位 | 覆盖（默认） | 运行状态、当前步骤 |
| 计数器、累计值 | 累加 (`add`) | 步骤数、token 用量 |

## 5. MessagesState：对话应用的快捷方式

构建对话应用时，几乎总需要一个追加式的 `messages` 列表。LangGraph 提供了内置的 `MessagesState`，不用自己定义。

In [ ]:
# 先看 MessagesState 的定义
from langgraph.graph import MessagesState
import inspect

print(inspect.getsource(MessagesState))

`MessagesState` 本质上就是一个带了 reducer 的 `TypedDict`，内部的 `messages` 字段使用了专门针对 LangChain Message 对象优化的 `add_messages` reducer。它比 `operator.add` 更智能：

- 自动处理 `AIMessage`、`HumanMessage` 等不同类型
- 支持按 `id` 去重更新（同 id 的消息会被替换而非重复追加）

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# 用 MessagesState 构建简单对话
def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

app = graph.compile()

result = app.invoke({"messages": [
    SystemMessage(content="你是一个简洁的助手，回答不超过两句话。"),
    HumanMessage(content="什么是 LangGraph？"),
]})

# 打印所有消息
for msg in result["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"[{role}] {msg.content}")

### 刚才发生了什么？

1. `MessagesState` 自带一个 `messages: Annotated[list, add_messages]` 字段。
2. 我们输入了 System + Human 两条消息，`chatbot` 节点调用模型后返回 `[AIMessage]`。
3. 由于 `add_messages` reducer 的存在，AI 的回复被**追加**到消息列表，而不是覆盖。
4. 最终 `messages` 包含了完整的对话历史：System → Human → AI。

## 6. 扩展 MessagesState

如果除了 `messages` 还需要其他字段，直接继承 `MessagesState`：

In [ ]:
# 继承 MessagesState 并添加自定义字段
class ChatState(MessagesState):
    user_name: str         # 覆盖
    turn_count: Annotated[int, add]  # 累加

def greet(state: ChatState) -> dict:
    name = state["user_name"]
    response = llm.invoke(
        state["messages"] + [HumanMessage(content=f"请用一句话和 {name} 打招呼")]
    )
    return {
        "messages": [response],
        "turn_count": 1,
    }

graph = StateGraph(ChatState)
graph.add_node("greet", greet)
graph.add_edge(START, "greet")
graph.add_edge("greet", END)

app = graph.compile()
result = app.invoke({
    "messages": [SystemMessage(content="你是一个热情的助手。")],
    "user_name": "小明",
    "turn_count": 0,
})

print(f"对话轮次: {result['turn_count']}")
print(f"最后一条消息: {result['messages'][-1].content}")

继承 `MessagesState` 是最常见的做法——你免费获得了消息追加能力，同时可以添加任何业务字段。

## 7. State 设计最佳实践

| 实践 | 说明 |
|------|------|
| 对话应用优先用 `MessagesState` | 内置 `add_messages` reducer，省去手动处理 |
| 需要额外字段时继承 `MessagesState` | 保持 messages 能力，扩展业务字段 |
| 字段语义决定 reducer | 历史类追加，状态类覆盖，计数类累加 |
| 保持 State 精简 | 只放节点间需要共享的数据，节点内部的临时变量不要放进 State |
| 为每个字段写清楚类型 | 方便 IDE 补全和错误排查 |

## 总结

本章学习了：
- ✅ State 的默认更新行为是覆盖
- ✅ 用 `Annotated[type, reducer]` 自定义更新策略
- ✅ `operator.add` 作为列表追加 / 整数累加的 reducer
- ✅ 编写自定义 reducer 函数（如去重追加）
- ✅ 同一个 State 中混合使用覆盖和追加
- ✅ `MessagesState` 的用法和继承扩展

## 下一章预告

**第三章：条件分支与循环** —— 到目前为止我们的图都是线性的（A → B → END）。下一章学习 `add_conditional_edges` 实现动态路由，以及如何构建带循环的图——这是实现 Agent「思考-行动-观察」循环的关键。